In [ ]:
import torch
from torch_geometric.nn import MetaPath2Vec
from torch_geometric.data import HeteroData

# Örnek heterojen veri seti oluşturma
data = HeteroData()

# Düğüm tipleri ve sayıları
data['author'].num_nodes = 100
data['paper'].num_nodes = 200
data['institution'].num_nodes = 50

data['author', 'writes', 'paper'].edge_index = torch.randint(0, 100, (2, 500))
data['author', 'friends', 'author'].edge_index = torch.randint(0, 20, (2, 100))
data['paper', 'written_by', 'author'].edge_index = data['author', 'writes', 'paper'].edge_index.flip(0)
data['author', 'affiliated_with', 'institution'].edge_index = torch.randint(0, 100, (2, 300))
data['institution', 'has_author', 'author'].edge_index = data['author', 'affiliated_with', 'institution'].edge_index.flip(0)

# Metapath tanımı: yazar -> makale -> yazar
metapath = [
    ('author', 'writes', 'paper'),
    ('paper', 'written_by', 'author'),
    ('author', 'friends', 'author'),
]

# Metapath2Vec modelinin tanımlanması
model = MetaPath2Vec(
    edge_index_dict=data.edge_index_dict,
    embedding_dim=16,
    metapath=metapath,
    walk_length=10,
    context_size=7,
    walks_per_node=5,
    num_negative_samples=5,
    sparse=True
)

# Modelin eğitimi için veri yükleyici
loader = model.loader(batch_size=128, shuffle=True, num_workers=4)
optimizer = torch.optim.SparseAdam(model.parameters(), lr=0.01)

# Eğitim döngüsü
def train():
    model.train()
    total_loss = 0
    for pos_rw, neg_rw in loader:
        optimizer.zero_grad()
        loss = model.loss(pos_rw, neg_rw)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

# Modeli eğit
for epoch in range(1, 2):
    loss = train()
    print(f'Epoch: {epoch}, Loss: {loss:.4f}')

# Yazar düğümleri için gömüleri elde etme
author_embeddings = model('author')

In [5]:
model('author').size()

torch.Size([100, 16])